### 1. Install Libraries

In [1]:
!pip install lightgbm scikit-learn

### 2. Zindi Platform Setup and Data Download

This section handles the integration with the Zindi platform. It installs the custom Zindi package, authenticates the user, allows selection of the specific challenge ('farm-to-feed-shopping-basket-recommendation-challenge'), and then downloads the relevant datasets (Train.csv, Test.csv, sku_data.csv, etc.) into the 'dataset' directory. This ensures all required data is available for analysis.

In [2]:
# Package to have access to the Zindi Platform features
!pip -q install git+https://github.com/eaedk/testing-zindi-package.git
from zindi.user import Zindian
#@title Input Username

# Login info for a Zindi Account

USERNAME = "keystats" #@param {type : "string"}
# object
user = Zindian(username=USERNAME) # Sign in
user.select_a_challenge(reward='all', kind='Competition', active='true')
user.which_challenge                                    # Get information about the selected challenge
user.download_dataset(destination="dataset") # Download the dataset of the selected challenge

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Your password
>> ··········

[ 🟢 ] 👋🏾👋🏾 Welcome keystats 👋🏾👋🏾

__________________________________________________________________________________________________________________________________
|     |              |                  |                    |          
|index|  challenge   |     problem      |       reward       |    id    
|     |              |                  |                    |          
----------------------------------------------------------------------------------------------------------------------------------
|  0  |Public Compet |    Prediction    |    $11 000 USD     | barbados-traffic-analysis-challenge...
----------------------------------------------------------------------------------------------------------------------------------
|  1  |Public Compet |                  |    €35 000 EUR     | the-ai-telco-troubleshooting-challenge...
------------------------------------

dataset/customer_data.csv: 23.1ko [00:00, 23.4Mo/s]
dataset/SampleSubmission.csv: 6.50Mo [00:01, 6.25Mo/s]
dataset/Test.csv: 25.9Mo [00:02, 12.5Mo/s]
dataset/sku_data.csv: 33.8ko [00:00, 15.2Mo/s]
dataset/Train.csv: 263Mo [00:14, 18.6Mo/s]
dataset/variable_description.pdf: 59.6ko [00:00, 455ko/s]


### 3. Import Essential Libraries

Here, all critical Python libraries are imported. This includes `pandas` and `numpy` for data manipulation, `lightgbm` for gradient boosting models, `os` for operating system interactions, `sklearn.decomposition` and `sklearn.cluster` for advanced feature engineering, `sklearn.preprocessing` for data scaling, and standard utilities like `warnings`, `random`, and `sys`.

In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import os
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
import random
import sys

### 4. Check CPU Count



In [ ]:
import multiprocessing
multiprocessing.cpu_count()

### 5. Define Global Configuration Parameters

This cell establishes global constants for the entire notebook. `RANDOM_SEED` ensures reproducibility across runs. `NUM_ENSEMBLE_FOLDS` and `ENSEMBLE_SEEDS` define the ensemble strategy. `FINAL_N_ESTIMATORS_CLF` and `FINAL_N_ESTIMATORS_REG` set the number of boosting rounds for the classifier and regressor models, respectively, which are crucial for model complexity and training time.

In [5]:
# --- GLOBAL CONFIGURATION ---
RANDOM_SEED = 42
NUM_ENSEMBLE_FOLDS = 5
ENSEMBLE_SEEDS = [42, 101, 202, 303, 404]

# 🛑 OPTIMIZED N_ESTIMATORS
FINAL_N_ESTIMATORS_CLF = 3500
FINAL_N_ESTIMATORS_REG = 1500


### 6. Setup Reproducibility and Suppress Warnings

This critical section enforces complete reproducibility across the entire execution. It sets environment variables for Python, NumPy, and various numerical libraries (`OMP_NUM_THREADS`, `MKL_NUM_THREADS`, etc.) to control parallel processing and ensure deterministic outcomes. It also suppresses warnings to keep the output clean. The `set_all_seeds` function is defined and called to ensure consistent random states for all operations.

In [6]:
# ============================================================
# CRITICAL: COMPLETE REPRODUCIBILITY SETUP
# ============================================================

# Set ALL environment variables for reproducibility
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
os.environ['OMP_NUM_THREADS'] = '32'
os.environ['MKL_NUM_THREADS'] = '32'
os.environ['OPENBLAS_NUM_THREADS'] = '32'
os.environ['VECLIB_MAXIMUM_THREADS'] = '32'
os.environ['NUMEXPR_NUM_THREADS'] = '32'
os.environ['LIGHTGBM_EXEC_GPU'] = 'false'

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Suppress warnings
warnings.filterwarnings('ignore')


def set_all_seeds(seed):
    """Sets seed for reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_all_seeds(RANDOM_SEED)

LOOKBACK_PERIODS = [2, 4, 8, 12, 16, 24]

### 7. Define Common LightGBM Settings

This cell defines a dictionary `LGBM_COMMON` containing shared configuration parameters for all LightGBM models used in the solution. These settings include `boosting_type`, `random_state`, `deterministic` flag, `num_threads`, and various seeds to ensure reproducible results for the LightGBM models.

In [7]:

# ============================================================
# LGBM SETTINGS WITH CORRECT PARAMETER NAMES
# ============================================================

LGBM_COMMON = {
    'boosting_type': 'gbdt',
    'random_state': RANDOM_SEED,
    'deterministic': True,
    'num_threads': -1,
    'verbose': -1,
    'force_row_wise': True,
    'bagging_seed': RANDOM_SEED,
    'feature_fraction_seed': RANDOM_SEED,
    'data_random_seed': RANDOM_SEED,
    'extra_seed': RANDOM_SEED,
    'seed': RANDOM_SEED,
    'n_jobs': -1,
}


### 8. Define Optimized Hyperparameters

This section outlines the specific, optimized hyperparameters for both the classification (CLF) and regression (REG) LightGBM models, separated for 1-week and 2-week predictions. These parameters (e.g., learning rate, num_leaves, max_depth, subsample, reg_alpha/lambda) are carefully tuned to maximize model performance and generalize well.

In [8]:

# ============================================================
# 0. OPTIMIZED Hyperparameters
# ============================================================


# 🎯 CLASSIFIER
CLF_1W_PARAMS = {
    'learning_rate': 0.008,
    'num_leaves': 255,
    'max_depth': 15,
    'min_child_samples': 220,
    'subsample': 0.65,
    'colsample_bytree': 0.5,
    'reg_alpha': 0.6,
    'reg_lambda': 0.6,
    'min_child_weight': 3.5,
}

# 🎯 WEEK 2
CLF_2W_PARAMS = {
    'learning_rate': 0.018,
    'num_leaves': 288,
    'max_depth': 15,
    'min_child_samples': 180,
    'subsample': 0.6,
    'colsample_bytree': 0.5,
    'reg_alpha': 0.3,
    'reg_lambda': 0.3,
    'min_child_weight': 4.0
}

# 🎯 REGRESSOR
REG_1W_PARAMS = {
    'learning_rate': 0.012,
    'num_leaves': 275,
    'max_depth': 14,
    'min_child_samples': 180,
    'subsample': 0.65,
    'colsample_bytree': 0.65,
    'reg_alpha': 1.2,
    'reg_lambda': 1.2,
    'min_child_weight': 0.07,
    'subsample_freq': 1
}

REG_2W_PARAMS = {
    'learning_rate': 0.018,
    'num_leaves': 245,
    'max_depth': 13,
    'min_child_samples': 240,
    'subsample': 0.55,
    'colsample_bytree': 0.55,
    'reg_alpha': 2.2,
    'reg_lambda': 2.2,
    'min_child_weight': 0.14,
    'subsample_freq': 1
}

### 9. Load Data and Initial Merges

This cell is responsible for loading the primary datasets (`Train.csv`, `Test.csv`, `sku_data.csv`) into pandas DataFrames. It then performs an initial merge to enrich the transaction data with product information from `sku_data`, specifically `product_name` and `grade_active_status`. Date columns (`week_start`, `customer_created_at`) are converted to datetime objects for proper temporal feature engineering.

In [9]:


# ============================================================
# 1. Data Loading & Initial Merges
# ============================================================

print("Loading data...")
df_train = pd.read_csv("dataset/Train.csv")
df_test = pd.read_csv("dataset/Test.csv")
sku_data = pd.read_csv("dataset/sku_data.csv")

# Add product info
sku_map = sku_data[
    ['product_unit_variant_id', 'product_name', 'grade_active_status']
].drop_duplicates()

df_train = df_train.merge(sku_map, on='product_unit_variant_id', how='left')
df_test = df_test.merge(sku_map, on='product_unit_variant_id', how='left')

# Convert date columns
DATE_COLS = ['week_start', 'customer_created_at']
for df in [df_train, df_test]:
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

print(f"Train size: {len(df_train)}, Test size: {len(df_test)}")

Loading data...
Train size: 2114436, Test size: 275796


### 10. Define Base Feature Engineering Functions

This section defines foundational feature engineering functions: `create_price_features` extracts price-related metrics and categorizes prices; `create_temporal_features` generates standard time-based features like week/month/day of year, customer tenure, and day of week; `create_seasonal_features` adds a product-week interaction feature. These functions create general-purpose features applicable to both classification and regression tasks.

In [10]:


# ============================================================
# 2. FIXED: ADVANCED Feature Engineering (DETERMINISTIC)
# ============================================================

def create_price_features(df):
    """Add price-related features with DETERMINISTIC binning."""
    if 'selling_price' in df.columns:
        product_avg_price = df.groupby('product_unit_variant_id')['selling_price'].transform('mean')
        df['price_ratio_to_avg'] = (df['selling_price'] - product_avg_price) / (product_avg_price + 1e-6)

        if len(df) > 0:
            try:
                sorted_prices = np.sort(df['selling_price'].values)
                q1 = sorted_prices[int(len(sorted_prices) * 0.25)]
                q2 = sorted_prices[int(len(sorted_prices) * 0.50)]
                q3 = sorted_prices[int(len(sorted_prices) * 0.75)]

                bins = [-np.inf, q1, q2, q3, np.inf]
                df['price_category'] = pd.cut(df['selling_price'], bins=bins, labels=False)
            except Exception as e:
                df['price_category'] = df['selling_price'].rank(method='first', pct=True)
                df['price_category'] = df['price_category'].apply(
                    lambda x: 0 if x < 0.25 else 1 if x < 0.5 else 2 if x < 0.75 else 3
                )
        else:
            df['price_category'] = 0

        df['price_category'] = df['price_category'].fillna(0).astype('category')
    return df

def create_temporal_features(df):
    """Creates basic temporal and tenure-based features."""
    df['week_of_year'] = df['week_start'].dt.isocalendar().week.astype(int)
    df['month'] = df['week_start'].dt.month
    df['day_of_year'] = df['week_start'].dt.dayofyear
    df['tenure_days'] = (df['week_start'] - df['customer_created_at']).dt.days
    df['tenure_weeks'] = df['tenure_days'] // 7
    df['day_of_week'] = df['week_start'].dt.dayofweek
    df['week_of_month'] = (df['week_start'].dt.day - 1) // 7 + 1

    df['customer_id'] = df['customer_id'].astype('category')
    df['product_unit_variant_id'] = df['product_unit_variant_id'].astype('category')
    return df

def create_seasonal_features(df):
    """Add seasonal interaction (RULE-COMPLIANT)."""
    if 'week_of_year' in df.columns and 'product_unit_variant_id' in df.columns:
        df['week_product_interaction'] = (
            df['week_of_year'].astype(str) + "_" +
            df['product_unit_variant_id'].astype(str)
        ).astype('category')
    return df

### 11. Create Regressor-Only Loyalty Features

This crucial function `create_regressor_loyalty_features` generates advanced loyalty-based features specifically designed for the regression models (quantity prediction). These include quantity stability metrics (mean, std, median, coefficient of variation), purchase cadence (inter-purchase days, regularity), seasonal quantity baselines, price-quantity elasticity, and quantity trend analysis. The goal is to stabilize MAE without impacting AUC by providing deeper insights into customer purchasing behavior.

In [11]:


# ============================================================
# 2.1 NEW: LOYALTY FEATURES ONLY FOR REGRESSORS
# ============================================================

def create_regressor_loyalty_features(df, source_df=None):
    """
    Create loyalty features EXCLUSIVELY for regressors (quantity prediction).
    These features are designed to stabilize MAE without affecting AUC.
    """
    print("  Creating REGRESSOR-ONLY loyalty features (for MAE stabilization)...")

    # Use source_df if provided (for test set), otherwise use df itself
    working_df = source_df if source_df is not None else df

    # Make a copy to avoid modifying original
    df_reg = df.copy()

    # Sort by customer and date for temporal analysis
    working_df_sorted = working_df.sort_values(['customer_id', 'product_unit_variant_id', 'week_start'])

    # 1. QUANTITY STABILITY FEATURES (GOLD FOR MAE)
    print("    - Quantity stability features...")

    # Calculate quantity consistency at customer-product level
    cp_qty_stats = working_df_sorted.groupby(['customer_id', 'product_unit_variant_id']).agg({
        'qty_this_week': ['mean', 'std', 'median', 'min', 'max', 'count'],
        'week_start': 'nunique'
    }).reset_index()

    cp_qty_stats.columns = ['customer_id', 'product_unit_variant_id',
                           'cp_qty_mean', 'cp_qty_std', 'cp_qty_median',
                           'cp_qty_min', 'cp_qty_max', 'cp_qty_count', 'cp_active_weeks']

    # Quantity consistency metrics
    cp_qty_stats['cp_qty_cv'] = cp_qty_stats['cp_qty_std'] / (cp_qty_stats['cp_qty_mean'] + 1e-6)
    cp_qty_stats['cp_qty_consistency'] = 1 - cp_qty_stats['cp_qty_cv'].clip(0, 1)
    cp_qty_stats['cp_qty_range_ratio'] = (cp_qty_stats['cp_qty_max'] - cp_qty_stats['cp_qty_min']) / (cp_qty_stats['cp_qty_mean'] + 1e-6)

    # Flag for highly stable quantity buyers
    cp_qty_stats['high_stability_flag'] = (
        (cp_qty_stats['cp_qty_count'] >= 3) &
        (cp_qty_stats['cp_qty_cv'] < 0.3)
    ).astype(int)

    # 2. PURCHASE CADENCE FOR QUANTITY PREDICTION
    print("    - Purchase cadence for quantity...")

    # Calculate inter-purchase intervals for quantity prediction
    purchase_events = working_df_sorted[working_df_sorted['purchased_this_week'] == 1].copy()

    if len(purchase_events) > 0:
        purchase_events['prev_purchase'] = purchase_events.groupby(
            ['customer_id', 'product_unit_variant_id']
        )['week_start'].shift(1)

        purchase_events['prev_qty'] = purchase_events.groupby(
            ['customer_id', 'product_unit_variant_id']
        )['qty_this_week'].shift(1)

        purchase_events['inter_purchase_days'] = (
            purchase_events['week_start'] - purchase_events['prev_purchase']
        ).dt.days

        # Quantity changes with time gap
        purchase_events['qty_time_ratio'] = purchase_events['qty_this_week'] / (purchase_events['inter_purchase_days'] + 1e-6)

        # Aggregate cadence-quantity relationships
        cadence_qty_stats = purchase_events.groupby(['customer_id', 'product_unit_variant_id']).agg({
            'inter_purchase_days': ['mean', 'std'],
            'qty_time_ratio': ['mean', 'std'],
            'qty_this_week': 'count'
        }).reset_index()

        cadence_qty_stats.columns = ['customer_id', 'product_unit_variant_id',
                                    'cp_ipd_mean', 'cp_ipd_std',
                                    'cp_qty_time_mean', 'cp_qty_time_std',
                                    'cp_purchase_count']

        # Merge cadence stats
        cp_qty_stats = cp_qty_stats.merge(cadence_qty_stats, on=['customer_id', 'product_unit_variant_id'], how='left')

        # Calculate cadence regularity
        cp_qty_stats['cp_cadence_regularity'] = 1 - (cp_qty_stats['cp_ipd_std'] / (cp_qty_stats['cp_ipd_mean'] + 1e-6))
        cp_qty_stats['cp_cadence_regularity'] = cp_qty_stats['cp_cadence_regularity'].clip(0, 1)

        # Fill NaN for customers/products with only one purchase
        cadence_cols = ['cp_ipd_mean', 'cp_ipd_std', 'cp_qty_time_mean', 'cp_qty_time_std', 'cp_cadence_regularity']
        cp_qty_stats[cadence_cols] = cp_qty_stats[cadence_cols].fillna(0)
    else:
        # Initialize cadence columns
        cadence_cols = ['cp_ipd_mean', 'cp_ipd_std', 'cp_qty_time_mean', 'cp_qty_time_std',
                       'cp_cadence_regularity', 'cp_purchase_count']
        for col in cadence_cols:
            cp_qty_stats[col] = 0

    # 3. SEASONAL QUANTITY BASELINES (MAE GOLD)
    print("    - Seasonal quantity baselines...")

    # Calculate monthly quantity patterns
    working_df_sorted['month'] = working_df_sorted['week_start'].dt.month
    working_df_sorted['quarter'] = working_df_sorted['week_start'].dt.quarter

    # Customer-product-month quantity baselines
    cp_seasonal_qty = working_df_sorted.groupby(['customer_id', 'product_unit_variant_id', 'month']).agg({
        'qty_this_week': ['mean', 'median', 'std', 'count']
    }).reset_index()

    cp_seasonal_qty.columns = ['customer_id', 'product_unit_variant_id', 'month',
                              'cp_month_qty_mean', 'cp_month_qty_median',
                              'cp_month_qty_std', 'cp_month_count']

    # Calculate seasonal consistency
    cp_seasonal_qty['cp_seasonal_cv'] = cp_seasonal_qty['cp_month_qty_std'] / (cp_seasonal_qty['cp_month_qty_mean'] + 1e-6)
    cp_seasonal_qty['cp_seasonal_stable'] = (cp_seasonal_qty['cp_month_count'] >= 2) & (cp_seasonal_qty['cp_seasonal_cv'] < 0.4)

    # 4. PRICE-QUANTITY RELATIONSHIPS FOR LOYAL CUSTOMERS
    print("    - Price-quantity relationships...")

    # Calculate price elasticity at customer-product level
    price_qty_corr = working_df_sorted.groupby(['customer_id', 'product_unit_variant_id']).apply(
        lambda x: np.corrcoef(x['selling_price'], x['qty_this_week'])[0, 1]
        if len(x) > 3 and x['selling_price'].std() > 0 and x['qty_this_week'].std() > 0 else 0
    ).reset_index(name='cp_price_elasticity')

    # Merge price elasticity
    cp_qty_stats = cp_qty_stats.merge(price_qty_corr, on=['customer_id', 'product_unit_variant_id'], how='left')
    cp_qty_stats['cp_price_elasticity'] = cp_qty_stats['cp_price_elasticity'].fillna(0)

    # Flag price-insensitive loyal customers
    cp_qty_stats['cp_price_insensitive'] = (
        (cp_qty_stats['cp_purchase_count'] >= 3) &
        (cp_qty_stats['cp_price_elasticity'] > -0.2)
    ).astype(int)

    # 5. QUANTITY TREND ANALYSIS
    print("    - Quantity trend analysis...")

    # Calculate quantity trends over time
    if len(working_df_sorted) > 0:
        from scipy.stats import linregress

        def calculate_qty_trend(group):
            if len(group) < 3:
                return 0, 0
            try:
                x = np.arange(len(group))
                y = group['qty_this_week'].values
                slope, intercept, r_value, p_value, std_err = linregress(x, y)
                return slope, r_value
            except:
                return 0, 0

        qty_trends = working_df_sorted.groupby(['customer_id', 'product_unit_variant_id']).apply(
            lambda x: calculate_qty_trend(x)
        ).reset_index(name='qty_trend_info')

        qty_trends[['cp_qty_trend', 'cp_qty_trend_corr']] = pd.DataFrame(
            qty_trends['qty_trend_info'].tolist(), index=qty_trends.index
        )

        cp_qty_stats = cp_qty_stats.merge(
            qty_trends[['customer_id', 'product_unit_variant_id', 'cp_qty_trend', 'cp_qty_trend_corr']],
            on=['customer_id', 'product_unit_variant_id'], how='left'
        )
        cp_qty_stats[['cp_qty_trend', 'cp_qty_trend_corr']] = cp_qty_stats[['cp_qty_trend', 'cp_qty_trend_corr']].fillna(0)
    else:
        cp_qty_stats['cp_qty_trend'] = 0
        cp_qty_stats['cp_qty_trend_corr'] = 0

    # 6. MERGE ALL REGRESSOR-ONLY LOYALTY FEATURES
    print("    - Merging regressor-only loyalty features...")

    # Merge quantity stability features
    df_reg = df_reg.merge(
        cp_qty_stats[['customer_id', 'product_unit_variant_id',
                     'cp_qty_mean', 'cp_qty_median', 'cp_qty_std',
                     'cp_qty_consistency', 'cp_qty_cv', 'high_stability_flag',
                     'cp_cadence_regularity', 'cp_price_elasticity',
                     'cp_price_insensitive', 'cp_qty_trend', 'cp_qty_trend_corr']],
        on=['customer_id', 'product_unit_variant_id'], how='left'
    )

    # Merge seasonal quantity features
    df_reg = df_reg.merge(
        cp_seasonal_qty[['customer_id', 'product_unit_variant_id', 'month',
                        'cp_month_qty_median', 'cp_seasonal_stable']],
        on=['customer_id', 'product_unit_variant_id', 'month'], how='left'
    )

    # 7. CREATE QUANTITY PREDICTION CONFIDENCE FEATURES
    print("    - Quantity prediction confidence...")

    # Expected quantity based on stability
    df_reg['expected_qty_stable'] = np.where(
        df_reg['high_stability_flag'] == 1,
        df_reg['cp_qty_median'],
        np.nan
    )

    # Expected quantity based on seasonality
    df_reg['expected_qty_seasonal'] = np.where(
        df_reg['cp_seasonal_stable'] == True,
        df_reg['cp_month_qty_median'],
        np.nan
    )

    # Combined expected quantity (prioritize stability, then seasonality)
    df_reg['expected_qty_combined'] = df_reg['expected_qty_stable'].combine_first(
        df_reg['expected_qty_seasonal']
    )

    # Quantity prediction confidence score
    df_reg['qty_pred_confidence'] = (
        df_reg['cp_qty_consistency'].fillna(0) * 0.4 +
        df_reg['cp_cadence_regularity'].fillna(0) * 0.3 +
        (df_reg['cp_seasonal_stable'].fillna(0).astype(float)) * 0.2 +
        (1 - df_reg['cp_qty_cv'].fillna(1).clip(0, 1)) * 0.1
    )

    # 8. FILL NA VALUES STRATEGICALLY

    # For stability metrics, fill with median values
    stability_cols = ['cp_qty_consistency', 'cp_cadence_regularity', 'qty_pred_confidence']
    for col in stability_cols:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].fillna(0.5)

    # For quantity values, fill with global median
    qty_cols = ['cp_qty_mean', 'cp_qty_median', 'cp_month_qty_median']
    global_qty_median = df_reg['qty_this_week'].median() if 'qty_this_week' in df_reg.columns else 1
    for col in qty_cols:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].fillna(global_qty_median)

    # For flags, fill with 0 (not stable)
    flag_cols = ['high_stability_flag', 'cp_price_insensitive']
    for col in flag_cols:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].fillna(0).astype(int)

    # For elasticities and trends, fill with 0 (neutral)
    numeric_cols = ['cp_price_elasticity', 'cp_qty_trend', 'cp_qty_trend_corr']
    for col in numeric_cols:
        if col in df_reg.columns:
            df_reg[col] = df_reg[col].fillna(0)

    print(f"    ✓ Added {len([c for c in df_reg.columns if 'cp_' in c or 'qty_' in c])} REGRESSOR-ONLY loyalty features")
    print(f"    Key features for MAE stabilization:")
    print(f"      - cp_qty_consistency: How consistent are quantities?")
    print(f"      - high_stability_flag: Flag for stable quantity buyers")
    print(f"      - cp_month_qty_median: Seasonal baseline for quantity")
    print(f"      - qty_pred_confidence: Confidence in quantity prediction")

    return df_reg

### 12. Define Historical Feature Engineering Functions

This section provides functions to generate historical aggregate features with strict leakage control. `generate_customer_product_history` computes statistics (sum, mean, count, recency) for each customer-product pair over specified lookback periods. `generate_product_popularity` calculates product-level purchase and quantity statistics. `generate_customer_total_history` aggregates customer-level purchase behavior. These functions are vital for capturing past trends without data leakage.

In [12]:


# ============================================================
# 3. HISTORICAL Feature Engineering (WITH STRICT LEAKAGE CONTROL)
# ============================================================

def generate_customer_product_history(df, max_week_date, lookback_weeks, exclude_target_cols=True):
    """Generate CP features with STRICT temporal filtering."""
    lookback_date = max_week_date - pd.Timedelta(weeks=lookback_weeks)

    history_df = df[(df['week_start'] < max_week_date) &
                    (df['week_start'] >= lookback_date)].copy()

    if exclude_target_cols:
        target_cols = [col for col in history_df.columns if col.startswith('Target_')]
        if target_cols:
            history_df = history_df.drop(columns=target_cols)

    agg_funcs = {
        'purchased_this_week': ['sum', 'mean', 'count'],
        'qty_this_week': ['sum', 'mean', 'std', 'max', 'median'],
        'spend_this_week': ['sum', 'mean', 'max'],
        'week_start': 'nunique'
    }

    cp_hist = history_df.groupby(['customer_id', 'product_unit_variant_id']).agg(agg_funcs)
    cp_hist.columns = [
        f'CP_{lookback_weeks}W_{col[0]}_{col[1]}' for col in cp_hist.columns
    ]

    recency_df = history_df[history_df['purchased_this_week'] == 1] \
        .groupby(['customer_id', 'product_unit_variant_id'])['week_start'] \
        .max().reset_index()

    RECENCY_COL_NAME = f'CP_{lookback_weeks}W_Recency_W'
    recency_df[RECENCY_COL_NAME] = (max_week_date - recency_df['week_start']).dt.days // 7

    cp_hist = cp_hist.merge(
        recency_df[['customer_id', 'product_unit_variant_id', RECENCY_COL_NAME]],
        on=['customer_id', 'product_unit_variant_id'],
        how='left'
    ).fillna({f'CP_{lookback_weeks}W_purchased_this_week_count': 0})

    return cp_hist

def generate_product_popularity(df, max_week_date, lookback_weeks, exclude_target_cols=True):
    """Generate product features with STRICT temporal filtering."""
    lookback_date = max_week_date - pd.Timedelta(weeks=lookback_weeks)

    history_df = df[(df['week_start'] < max_week_date) &
                    (df['week_start'] >= lookback_date)].copy()

    if exclude_target_cols:
        target_cols = [col for col in history_df.columns if col.startswith('Target_')]
        if target_cols:
            history_df = history_df.drop(columns=target_cols)

    agg_funcs = {
        'purchased_this_week': ['sum', 'mean'],
        'qty_this_week': ['sum', 'mean', 'median'],
        'customer_id': 'nunique'
    }

    prod_pop = history_df.groupby('product_unit_variant_id').agg(agg_funcs)
    prod_pop.columns = [
        f'P_{lookback_weeks}W_{col[0]}_{col[1]}' for col in prod_pop.columns
    ]
    return prod_pop

def generate_customer_total_history(df, max_week_date, lookback_weeks, exclude_target_cols=True):
    """Generate customer features with STRICT temporal filtering."""
    lookback_date = max_week_date - pd.Timedelta(weeks=lookback_weeks)

    history_df = df[(df['week_start'] < max_week_date) &
                    (df['week_start'] >= lookback_date)].copy()

    if exclude_target_cols:
        target_cols = [col for col in history_df.columns if col.startswith('Target_')]
        if target_cols:
            history_df = history_df.drop(columns=target_cols)

    agg_funcs = {
        'purchased_this_week': ['sum', 'count', 'mean'],
        'qty_this_week': ['sum', 'mean', 'max', 'median'],
        'spend_this_week': ['sum', 'mean'],
        'product_unit_variant_id': 'nunique'
    }

    cust_hist = history_df.groupby('customer_id').agg(agg_funcs)
    cust_hist.columns = [
        f'C_{lookback_weeks}W_{col[0]}_{col[1]}' for col in cust_hist.columns
    ]
    cust_hist = cust_hist.reset_index()

    recency_df = history_df[history_df['purchased_this_week'] == 1] \
        .groupby('customer_id')['week_start'] \
        .max().reset_index()

    RECENCY_COL_NAME = f'C_{lookback_weeks}W_Recency_W'
    recency_df[RECENCY_COL_NAME] = (max_week_date - recency_df['week_start']).dt.days // 7

    cust_hist = cust_hist.merge(
        recency_df[['customer_id', RECENCY_COL_NAME]],
        on='customer_id',
        how='left'
    )

    cols_to_fill_zero = [col for col in cust_hist.columns if col not in ['customer_id', 'product_unit_variant_id']]
    cust_hist[cols_to_fill_zero] = cust_hist[cols_to_fill_zero].fillna(0)

    return cust_hist


### 13. Create Cross-Attention Features

The `create_cross_attention_features` function generates interaction features specifically for the regressor models. These features aim to capture nuanced relationships between customer recency, product popularity, and seasonal purchasing patterns. Examples include `product_attention_score`, `is_top3_product_this_week`, `competitor_density`, and `purchase_decision_score`, designed to provide high-signal information for quantity prediction.

In [13]:

# ============================================================
# 3.1 CREATIVE FEATURE ENGINEERING: Cross-Attention Features
# ============================================================

def create_cross_attention_features(df):
    """Create high-signal interaction features for REGRESSORS only."""

    print("  Creating cross-attention features (for regressors)...")

    cp_recency_cols = [col for col in df.columns if 'CP_' in col and 'Recency_W' in col]
    c_recency_cols = [col for col in df.columns if 'C_' in col and 'Recency_W' in col]
    p_purchase_cols = [col for col in df.columns if 'P_' in col and 'purchased_this_week_mean' in col]
    c_purchase_cols = [col for col in df.columns if 'C_' in col and 'purchased_this_week_mean' in col]

    if cp_recency_cols:
        cp_recency_col = cp_recency_cols[0]
        df['cust_avg_recency'] = df.groupby('customer_id')[cp_recency_col].transform('mean')
        df['product_attention_score'] = (df['cust_avg_recency'] - df[cp_recency_col]) / (df['cust_avg_recency'] + 1e-6)

    if p_purchase_cols and 'week_start' in df.columns:
        p_pop_col = next((col for col in p_purchase_cols if '12W' in col or '8W' in col), p_purchase_cols[0])
        weekly_top = df.groupby(['week_start', 'product_unit_variant_id'])[p_pop_col].transform('rank', method='dense')
        df['is_top3_product_this_week'] = (weekly_top <= 3).astype(int)
        df['competitor_density'] = df.groupby(['week_start'])[p_pop_col].transform(
            lambda x: (x > x.median()).sum() if len(x) > 1 else 0
        )

    if c_purchase_cols and p_purchase_cols and 'week_of_year' in df.columns:
        c_pop_col = next((col for col in c_purchase_cols if '8W' in col or '12W' in col), c_purchase_cols[0])
        p_pop_col = next((col for col in p_purchase_cols if '8W' in col or '12W' in col), p_purchase_cols[0])

        p_min = df[p_pop_col].min()
        p_max = df[p_pop_col].max()
        if p_max > p_min:
            df['product_trend_normalized'] = (df[p_pop_col] - p_min) / (p_max - p_min + 1e-6)
        else:
            df['product_trend_normalized'] = 0.5

        if 'purchased_this_week' in df.columns:
            seasonal_data = df[['product_unit_variant_id', 'week_of_year', 'purchased_this_week']].copy()
            season_map = seasonal_data.groupby(['product_unit_variant_id', 'week_of_year'])['purchased_this_week'].mean().reset_index()
            season_map.columns = ['product_unit_variant_id', 'week_of_year', 'product_seasonal_factor']
            df = df.merge(season_map, on=['product_unit_variant_id', 'week_of_year'], how='left')
            global_avg = df['purchased_this_week'].mean() if 'purchased_this_week' in df.columns else 0.5
            df['product_seasonal_factor'] = df['product_seasonal_factor'].fillna(global_avg)

        if 'product_seasonal_factor' in df.columns:
            df['purchase_decision_score'] = (
                df[c_pop_col].fillna(0) * 0.3 +
                df['product_trend_normalized'] * 0.4 +
                df['product_seasonal_factor'] * 0.3
            )

    return df

### 14. Add Weekly Delta Features

The `add_weekly_delta_features` function introduces week-over-week change features, primarily for regressor models. It calculates rolling averages of past purchases (e.g., `purchase_rolling_1w`, `2w`, `4w`) and uses these to derive 'momentum' features. It also tracks product-level trends, capturing how product popularity changes over time. These features help model recent shifts in purchasing behavior.

In [14]:
# ============================================================
# 3.2 CREATIVE FEATURE ENGINEERING: Weekly Delta Features
# ============================================================

def add_weekly_delta_features(df, source_df):
    """Add week-over-week change features for REGRESSORS only."""

    print("  Creating weekly delta features (for regressors)...")

    df = df.copy()
    purchase_matrix = source_df[['customer_id', 'product_unit_variant_id', 'week_start', 'purchased_this_week']].copy()
    purchase_matrix = purchase_matrix.sort_values(['customer_id', 'product_unit_variant_id', 'week_start'])

    purchase_matrix['purchase_rolling_1w'] = purchase_matrix.groupby(
        ['customer_id', 'product_unit_variant_id']
    )['purchased_this_week'].transform(
        lambda x: x.rolling(1, min_periods=1).mean().shift(1)
    )

    purchase_matrix['purchase_rolling_2w'] = purchase_matrix.groupby(
        ['customer_id', 'product_unit_variant_id']
    )['purchased_this_week'].transform(
        lambda x: x.rolling(2, min_periods=1).mean().shift(1)
    )

    purchase_matrix['purchase_rolling_4w'] = purchase_matrix.groupby(
        ['customer_id', 'product_unit_variant_id']
    )['purchased_this_week'].transform(
        lambda x: x.rolling(4, min_periods=1).mean().shift(1)
    )

    merge_cols = ['customer_id', 'product_unit_variant_id', 'week_start']
    delta_features = purchase_matrix[merge_cols + ['purchase_rolling_1w', 'purchase_rolling_2w', 'purchase_rolling_4w']]
    delta_features = delta_features.drop_duplicates(subset=merge_cols, keep='last')

    df = df.merge(delta_features, on=merge_cols, how='left')

    if 'purchased_this_week' in df.columns:
        df['purchase_momentum_1w'] = df['purchased_this_week'] - df['purchase_rolling_1w']
        df['purchase_momentum_2w'] = df['purchased_this_week'] - df['purchase_rolling_2w']
        df['purchase_momentum_4w'] = df['purchased_this_week'] - df['purchase_rolling_4w']
    else:
        df['purchase_momentum_1w'] = df['purchase_rolling_1w']
        df['purchase_momentum_2w'] = df['purchase_rolling_2w']
        df['purchase_momentum_4w'] = df['purchase_rolling_4w']

    if 'week_start' in df.columns and 'product_unit_variant_id' in df.columns:
        product_trend_data = purchase_matrix.sort_values(['product_unit_variant_id', 'week_start']).copy()
        product_trend_data['product_rolling_2w'] = product_trend_data.groupby('product_unit_variant_id')['purchased_this_week'].transform(
            lambda x: x.rolling(2, min_periods=1).mean()
        )

        product_trend_data['product_rolling_4w'] = product_trend_data.groupby('product_unit_variant_id')['purchased_this_week'].transform(
            lambda x: x.rolling(4, min_periods=1).mean()
        )

        product_trend_data['product_trend_2w'] = product_trend_data.groupby('product_unit_variant_id')['product_rolling_2w'].pct_change(1)
        product_trend_data['product_trend_4w'] = product_trend_data.groupby('product_unit_variant_id')['product_rolling_4w'].pct_change(1)

        product_features = product_trend_data[['product_unit_variant_id', 'week_start',
                                              'product_trend_2w', 'product_trend_4w']].drop_duplicates()
        df = df.merge(product_features, on=['product_unit_variant_id', 'week_start'], how='left')

    delta_cols = [col for col in df.columns if 'purchase_' in col or 'product_trend_' in col or 'product_rolling_' in col]
    for col in delta_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    print(f"    ✓ Added {len(delta_cols)} delta/trend features")
    return df

### 15. Create Behavioral Clustering Features

The `create_behavioral_clusters` function groups customers into behavioral segments using K-Means clustering. It leverages various historical features (e.g., purchase frequency, product variety, recency) to create clusters, and then assigns a `behavioral_cluster` ID and `cluster_distance` to each customer. This helps in understanding customer segments and their purchasing patterns, especially useful for regression models.

In [15]:
# ============================================================
# 3.3 CREATIVE FEATURE ENGINEERING: Behavioral Clustering
# ============================================================

def create_behavioral_clusters(df, n_clusters=8):
    """Cluster customers by behavioral patterns for REGRESSORS only."""

    print("  Creating behavioral clusters (for regressors)...")

    behavioral_features = []

    week_count_cols = [col for col in df.columns if 'C_' in col and 'week_start_nunique' in col]
    if week_count_cols:
        behavioral_features.append(week_count_cols[0])

    product_variety_cols = [col for col in df.columns if 'C_' in col and 'product_unit_variant_id_nunique' in col]
    if product_variety_cols:
        behavioral_features.append(product_variety_cols[0])

    recency_cols = [col for col in df.columns if 'C_' in col and 'Recency_W' in col]
    if recency_cols:
        behavioral_features.append(recency_cols[0])

    purchase_freq_cols = [col for col in df.columns if 'C_' in col and 'purchased_this_week_mean' in col]
    if purchase_freq_cols:
        medium_freq = next((col for col in purchase_freq_cols if '8W' in col or '12W' in col), purchase_freq_cols[0])
        behavioral_features.append(medium_freq)

    if behavioral_features and len(behavioral_features) >= 2:
        print(f"    Using {len(behavioral_features)} behavioral features: {behavioral_features}")

        X_behavior = df[behavioral_features].fillna(0).values
        unique_customers = df['customer_id'].nunique()

        if unique_customers >= n_clusters * 3:
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_behavior)
            kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10)
            df['behavioral_cluster'] = kmeans.fit_predict(X_scaled)
            df['behavioral_cluster'] = df['behavioral_cluster'].astype('category')
            df['cluster_distance'] = np.min(kmeans.transform(X_scaled), axis=1)

            for i in range(min(3, n_clusters)):
                df[f'distance_to_cluster_{i}'] = kmeans.transform(X_scaled)[:, i]

            print(f"    ✓ Created {n_clusters} behavioral clusters with distance features")
        else:
            print(f"    ⚠️ Not enough unique customers ({unique_customers}) for {n_clusters} clusters")
            df['behavioral_cluster'] = 0
            df['cluster_distance'] = 0
    else:
        print(f"    ⚠️ Insufficient behavioral features found ({len(behavioral_features)} available)")
        df['behavioral_cluster'] = 0
        df['cluster_distance'] = 0

    return df

### 16. Create Rolling SVD Features

The `create_rolling_svd_features` function generates Singular Value Decomposition (SVD) embeddings for customer-product interactions. It creates these features using rolling windows to capture evolving relationships over time, specifically designed to boost AUC. It computes customer and product embeddings and an 'svd_dot_product' as an interaction score, providing rich representations of user-item preferences.

In [16]:


# ============================================================
# 3.4 FIXED: ROLLING SVD FEATURES (DETERMINISTIC + AUC BOOSTED)
# ============================================================

def create_rolling_svd_features(df_train, df_test):
    """
    Create SVD features using ROLLING windows with DETERMINISTIC algorithm.
    Enhanced with Dot Product (Match Score) to maximize AUC.
    """
    print("\n=== CREATING ROLLING SVD FEATURES ===")

    unique_weeks = sorted(df_train['week_start'].unique())
    svd_train_dfs = []

    last_customer_emb_df = None
    last_product_emb_df = None

    for week_idx, current_week in enumerate(unique_weeks):
        if week_idx < 8:
            continue

        historical_data = df_train[df_train['week_start'] < current_week].copy()
        if len(historical_data) < 100:
            continue

        print(f"  Processing week {current_week.date()} (Week {week_idx + 1}/{len(unique_weeks)})")

        try:
            cp_matrix = historical_data.pivot_table(
                index='customer_id',
                columns='product_unit_variant_id',
                values='purchased_this_week',
                aggfunc='mean',
                fill_value=0
            )

            n_components = min(12, min(cp_matrix.shape) - 1)
            if n_components < 2: continue

            svd = TruncatedSVD(
                n_components=n_components,
                random_state=RANDOM_SEED,
                algorithm='arpack'
            )

            customer_embeddings = svd.fit_transform(cp_matrix)
            product_embeddings = svd.components_.T

            cust_cols = [f'cust_svd_{i}' for i in range(n_components)]
            prod_cols = [f'prod_svd_{i}' for i in range(n_components)]

            customer_emb_df = pd.DataFrame(customer_embeddings, index=cp_matrix.index, columns=cust_cols).reset_index()
            product_emb_df = pd.DataFrame(product_embeddings, index=cp_matrix.columns, columns=prod_cols).reset_index()

            def calculate_enhanced_svd(df_chunk, c_emb, p_emb):
                df_chunk = df_chunk.merge(c_emb, on='customer_id', how='left')
                df_chunk = df_chunk.merge(p_emb, on='product_unit_variant_id', how='left')

                dot_val = np.zeros(len(df_chunk))
                for i in range(n_components):
                    dot_val += df_chunk[f'cust_svd_{i}'].fillna(0) * df_chunk[f'prod_svd_{i}'].fillna(0)

                df_chunk['svd_dot_product'] = dot_val
                df_chunk['cust_svd_norm'] = np.linalg.norm(df_chunk[cust_cols].fillna(0).values, axis=1)
                df_chunk['prod_svd_norm'] = np.linalg.norm(df_chunk[prod_cols].fillna(0).values, axis=1)

                return df_chunk

            week_train_data = df_train[df_train['week_start'] == current_week][['customer_id', 'product_unit_variant_id']].copy()
            week_train_data = calculate_enhanced_svd(week_train_data, customer_emb_df, product_emb_df)
            week_train_data['week_start'] = current_week
            svd_train_dfs.append(week_train_data)

            last_customer_emb_df = customer_emb_df
            last_product_emb_df = product_emb_df

        except Exception as e:
            print(f"    Error processing week {current_week}: {e}")
            continue

    if svd_train_dfs:
        all_svd_train = pd.concat(svd_train_dfs, ignore_index=True)

        if last_customer_emb_df is not None:
            test_svd_data = df_test[['customer_id', 'product_unit_variant_id']].copy()
            test_svd_data = calculate_enhanced_svd(test_svd_data, last_customer_emb_df, last_product_emb_df)

            print(f"  Success: Added Dot Product and {n_components*2} SVD features.")
            return all_svd_train, test_svd_data

    return None, None


### 17. Create Cyclical Time Features

The `create_cyclical_features` function transforms temporal features (day of week, week of month, month, week of year) into cyclical representations using sine and cosine functions. This approach helps models understand the cyclic nature of these features (e.g., Sunday follows Saturday, December follows November) without imposing an artificial linear order, which can improve model performance.

In [17]:

# ============================================================
# 3.5 CYCLICAL TIME FEATURES
# ============================================================

def create_cyclical_features(df):
    """Create cyclical time features using sin/cos transforms."""
    print("  Creating cyclical time features...")

    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['week_of_month_sin'] = np.sin(2 * np.pi * df['week_of_month'] / 4)
    df['week_of_month_cos'] = np.cos(2 * np.pi * df['week_of_month'] / 4)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['week_of_year_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 52)
    df['week_of_year_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 52)

    return df

### 18. Main Feature Engineering Pipeline

This section orchestrates the application of all defined feature engineering functions. It first creates base price, temporal, and seasonal features. Then, it iteratively adds historical customer, product, and customer-product features using various lookback periods. Finally, it integrates cyclical time features and the rolling SVD features into the main datasets. This comprehensive pipeline builds the full feature set for model training.

In [ ]:


# ============================================================
# 4. MAIN PROCESSING PIPELINE WITH FEATURE SEGREGATION
# ============================================================

print("\n=== CREATING BASE FEATURES ===")
df_train = create_price_features(df_train)
df_test = create_price_features(df_test)

df_train = create_temporal_features(df_train)
df_test = create_temporal_features(df_test)

df_train = create_seasonal_features(df_train)
df_test = create_seasonal_features(df_test)

print("\n=== ADDING HISTORICAL FEATURES ===")
def add_historical_features_enhanced(df, source_df, lookback_periods):
    """Add historical features with STRICT leakage control."""
    df_list = []
    unique_weeks = df['week_start'].sort_values().unique()

    safe_source = source_df.drop(
        columns=[col for col in source_df.columns if col.startswith('Target_')],
        errors='ignore'
    ).copy()

    for week_date in unique_weeks:
        current_week_df = df[df['week_start'] == week_date].copy()

        for lb in lookback_periods:
            cp_features = generate_customer_product_history(safe_source, week_date, lb, exclude_target_cols=True)
            if not cp_features.empty:
                current_week_df = current_week_df.merge(
                    cp_features, on=['customer_id', 'product_unit_variant_id'], how='left')

            prod_features = generate_product_popularity(safe_source, week_date, lb, exclude_target_cols=True)
            if not prod_features.empty:
                current_week_df = current_week_df.merge(
                    prod_features, on='product_unit_variant_id', how='left')

            cust_features = generate_customer_total_history(safe_source, week_date, lb, exclude_target_cols=True)
            if not cust_features.empty:
                current_week_df = current_week_df.merge(
                    cust_features, on='customer_id', how='left')

        df_list.append(current_week_df)

    return pd.concat(df_list, ignore_index=True)

# Create base features for both classifier and regressor
df_train_fe_base = add_historical_features_enhanced(df_train, df_train, lookback_periods=LOOKBACK_PERIODS)
df_test_fe_base = add_historical_features_enhanced(df_test, df_train, lookback_periods=LOOKBACK_PERIODS)

print("\n=== ADDING CYCLICAL FEATURES ===")
df_train_fe_base = create_cyclical_features(df_train_fe_base)
df_test_fe_base = create_cyclical_features(df_test_fe_base)

print("\n=== ADDING ROLLING SVD FEATURES ===")
train_svd_df, test_svd_df = create_rolling_svd_features(df_train, df_test)

if train_svd_df is not None:
    df_train_fe_base = df_train_fe_base.merge(train_svd_df, on=['customer_id', 'product_unit_variant_id', 'week_start'], how='left')
    df_test_fe_base = df_test_fe_base.merge(test_svd_df, on=['customer_id', 'product_unit_variant_id'], how='left')
    print(f"  Successfully added rolling SVD features")
else:
    print("  Could not create rolling SVD features (insufficient data)")

# Fill NaN for SVD features
svd_cols = [col for col in df_train_fe_base.columns if 'svd_' in col]
for col in svd_cols:
    if col in df_test_fe_base.columns:
        df_test_fe_base[col] = df_test_fe_base[col].fillna(0)
    if col in df_train_fe_base.columns:
        df_train_fe_base[col] = df_train_fe_base[col].fillna(0)


### 19. Create Separate Datasets: Classifier vs Regressor

This cell separates the processed `df_train_fe_base` and `df_test_fe_base` into distinct DataFrames: `df_train_fe_clf`/`df_test_fe_clf` for the classifier, and `df_train_fe_reg`/`df_test_fe_reg` for the regressor. Critically, it then applies the regressor-specific loyalty, cross-attention, weekly delta, and behavioral clustering features *only* to the regressor datasets. This implements the key strategy of feature segregation for optimized AUC and MAE.

In [ ]:

# ============================================================
# 4.1 CREATE SEPARATE DATASETS: Classifier vs Regressor
# ============================================================

print("\n=== CREATING SEPARATE DATASETS ===")
print("Step 1: Creating CLASSIFIER dataset (no loyalty features)...")
df_train_fe_clf = df_train_fe_base.copy()
df_test_fe_clf = df_test_fe_base.copy()

print("Step 2: Creating REGRESSOR dataset with loyalty features...")
df_train_fe_reg = df_train_fe_base.copy()
df_test_fe_reg = df_test_fe_base.copy()

# Add regressor-only loyalty features
print("\n=== ADDING REGRESSOR-ONLY LOYALTY FEATURES ===")
df_train_fe_reg = create_regressor_loyalty_features(df_train_fe_reg, source_df=df_train)
df_test_fe_reg = create_regressor_loyalty_features(df_test_fe_reg, source_df=df_train)

print("\nStep 3: Adding Cross-Attention Features (Regressor-only)")
df_train_fe_reg = create_cross_attention_features(df_train_fe_reg)
df_test_fe_reg = create_cross_attention_features(df_test_fe_reg)

print("\nStep 4: Adding Weekly Delta Features (Regressor-only)")
df_train_fe_reg = add_weekly_delta_features(df_train_fe_reg, df_train)
df_test_fe_reg = add_weekly_delta_features(df_test_fe_reg, df_train)

print("\nStep 5: Adding Behavioral Clustering (Regressor-only)")
df_train_fe_reg = create_behavioral_clusters(df_train_fe_reg, n_clusters=8)
df_test_fe_reg = create_behavioral_clusters(df_test_fe_reg, n_clusters=8)


### 20. Handle Feature Segregation and Object Columns

This section prepares the feature sets for model training. It defines columns to be excluded and converts object type columns (e.g., product names, customer categories) into `Categorical` type, ensuring consistent categories across train and test sets. It then identifies numerical and categorical features for both the classifier and regressor datasets, paving the way for the final feature selection.

In [ ]:

# ============================================================
# 5. FEATURE SEGREGATION: Different Features for Classifiers vs Regressors
# ============================================================

EXCLUDE_COLS = [
    'ID', 'week_start', 'customer_created_at', 'qty_this_week',
    'num_orders_week', 'spend_this_week', 'purchased_this_week',
    'selling_price', 'cust_avg_recency'
] + [col for col in df_train.columns if col.startswith('Target_')]

print("\n=== HANDLING OBJECT COLUMNS ===")
object_columns = ['grade_name', 'unit_name', 'product_name', 'customer_category', 'customer_status']
for df_list in [[df_train_fe_clf, df_test_fe_clf], [df_train_fe_reg, df_test_fe_reg]]:
    for col in object_columns:
        if col in df_list[0].columns:
            df_list[0][col] = df_list[0][col].astype(str)
            df_list[1][col] = df_list[1][col].astype(str)

            all_categories = sorted(df_list[0][col].dropna().unique().tolist())
            if col in df_list[1].columns:
                test_categories = df_list[1][col].dropna().unique().tolist()
                for cat in test_categories:
                    if cat not in all_categories:
                        all_categories.append(cat)

            df_list[0][col] = pd.Categorical(df_list[0][col], categories=all_categories)
            df_list[1][col] = pd.Categorical(df_list[1][col], categories=all_categories)

print("\n=== CREATING SEPARATE FEATURE SETS ===")

# Get base features for classifier
numerical_cols_clf = df_train_fe_clf.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols_clf = df_train_fe_clf.select_dtypes(include=['category']).columns.tolist()

# Get enhanced features for regressor (including loyalty features)
numerical_cols_reg = df_train_fe_reg.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols_reg = df_train_fe_reg.select_dtypes(include=['category']).columns.tolist()

### 21. Define Final Feature Sets for Classifiers and Regressors

This cell implements the core strategy of using segregated feature sets. `CLASSIFIER_FEATURES` are carefully curated to be 'clean' (excluding loyalty and regressor-specific features) to maximize AUC. `REGRESSOR_FEATURES` include the full set, specifically retaining all the creative and loyalty-focused features designed to optimize MAE. This ensures each model receives the most appropriate features for its objective.

In [ ]:

# ============================================================
# 🎯 KEY STRATEGY: SEPARATE FEATURE SETS - LOYALTY ONLY FOR REGRESSORS
# ============================================================

# CLASSIFIER FEATURES: Clean, purchase-focused (for AUC) - NO LOYALTY FEATURES
CLASSIFIER_FEATURES = [
    col for col in (numerical_cols_clf + categorical_cols_clf)
    if col not in EXCLUDE_COLS and col != 'last_week_product'
    # Exclude all loyalty and regressor-specific features
    and not any(x in col for x in [
        'attention_score', 'decision_score', 'momentum_', 'trend_',
        'rolling_', 'cluster_distance', 'distance_to_cluster',
        'behavioral_cluster', 'competitor_density', 'is_top3',
        # Exclude loyalty features
        'cp_', 'loyal', 'cadence', 'elasticity', 'stability',
        'qty_pred', 'expected_qty', 'high_stability', 'price_insensitive',
        'month_qty', 'seasonal_stable', 'qty_consistency', 'qty_cv'
    ])
]
CLASSIFIER_FEATURES = list(set(CLASSIFIER_FEATURES))

# REGRESSOR FEATURES: Everything including creative features + LOYALTY FEATURES (for MAE)
REGRESSOR_FEATURES = [
    col for col in (numerical_cols_reg + categorical_cols_reg)
    if col not in EXCLUDE_COLS and col != 'last_week_product'
    # Include loyalty features (cp_, qty_, stability, etc.)
]
REGRESSOR_FEATURES = list(set(REGRESSOR_FEATURES))

print(f"Classifier features: {len(CLASSIFIER_FEATURES)} (clean set for AUC, NO loyalty)")
print(f"Regressor features: {len(REGRESSOR_FEATURES)} (full set + LOYALTY for MAE)")
print(f"Loyalty features for regressors only: {len([f for f in REGRESSOR_FEATURES if 'cp_' in f or 'qty_' in f])}")

# Sort by week_start for temporal consistency
df_full_train_clf = df_train_fe_clf.sort_values(by=['week_start', 'ID']).reset_index(drop=True)
df_full_train_reg = df_train_fe_reg.sort_values(by=['week_start', 'ID']).reset_index(drop=True)


# Calculate maximum quantity for clipping
MAX_QTY_TRAIN = df_train['qty_this_week'].max()
print(f"Maximum quantity in training data: {MAX_QTY_TRAIN}")

### 22. Define Reproducible LightGBM Training Function

This function `train_reproducible_lgbm` encapsulates the training logic for both LightGBM classifiers and regressors. It takes the DataFrame, target column, model type, parameters, and random seed. It applies specific `model_params` for reproducibility and sets `objective` and `metric` based on whether it's a classifier (binary, AUC) or regressor (regression_l1, MAE). For regressors, it filters training data to only include historical purchases and uses the full feature set, while classifiers use the clean feature set.

In [ ]:


# ============================================================
# 6. REPRODUCIBLE Model Training with FEATURE SEGREGATION
# ============================================================

def train_reproducible_lgbm(df_train, target_col, is_classifier, params, seed,
                           classifier_features, regressor_features):
    """Train 100% reproducible LightGBM model with feature segregation."""

    model_params = {
        'random_state': seed,
        'deterministic': True,
        'num_threads': -1,
        'verbose': -1,
        'force_row_wise': True,
        'bagging_seed': seed,
        'feature_fraction_seed': seed,
        'data_random_seed': seed,
        'extra_seed': seed,
        'seed': seed,
        'n_jobs': -1,
    }

    model_params.update(params)

    if is_classifier:
        # CLASSIFIER: Use CLEAN features only (NO loyalty features)
        X_train, y_train = df_train[classifier_features], df_train[target_col]

        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale_pos_weight = neg_count / max(pos_count, 1)

        model_params.update({
            'objective': 'binary',
            'metric': 'auc',
            'scale_pos_weight': scale_pos_weight,
            'n_estimators': FINAL_N_ESTIMATORS_CLF
        })

        model = lgb.LGBMClassifier(**model_params)
        print(f"  Training Classifier: {target_col} (N={FINAL_N_ESTIMATORS_CLF}, Features={len(classifier_features)}, NO loyalty)")

        model.fit(
            X_train, y_train,
            callbacks=[
                lgb.log_evaluation(0),
                lgb.reset_parameter(seed=lambda current_iter: seed)
            ]
        )

    else:
        # REGRESSOR: Use ALL features including LOYALTY FEATURES (for MAE)
        # Filter based on historical purchases
        df_train['historical_purchase_count'] = df_train.groupby(
            ['customer_id', 'product_unit_variant_id']
        )['purchased_this_week'].transform('sum')

        df_train_reg = df_train[df_train['historical_purchase_count'] > 0].copy()

        X_train, y_train = df_train_reg[regressor_features], df_train_reg[target_col]

        df_train.drop(columns=['historical_purchase_count'], errors='ignore', inplace=True)

        model_params.update({
            'objective': 'regression_l1',
            'metric': 'mae',
            'n_estimators': FINAL_N_ESTIMATORS_REG
        })

        model = lgb.LGBMRegressor(**model_params)
        print(f"  Training Regressor: {target_col} (N={FINAL_N_ESTIMATORS_REG}, Features={len(regressor_features)}, WITH loyalty)")

        model.fit(
            X_train, y_train,
            callbacks=[
                lgb.log_evaluation(0),
                lgb.reset_parameter(seed=lambda current_iter: seed)
            ]
        )

    return model


### 23. Execute Ensemble Training with Feature Segregation

This is the main model training block. It iterates through `NUM_ENSEMBLE_FOLDS` (defined by `ENSEMBLE_SEEDS`), training separate 1-week and 2-week classifiers and regressors for each fold. Crucially, it ensures feature consistency between train and test sets and applies the predefined `CLASSIFIER_FEATURES` to classifiers and `REGRESSOR_FEATURES` (including loyalty) to regressors. Predictions from each fold are collected and then averaged to form the final ensemble predictions, enhancing robustness.

In [ ]:

# ============================================================
# 7. ENSEMBLE TRAINING WITH FEATURE SEGREGATION
# ============================================================

print(f"\n=== Starting {NUM_ENSEMBLE_FOLDS}-Fold Ensemble Training ===")
print(f"=== Classifiers: {FINAL_N_ESTIMATORS_CLF} est. (Clean Features, NO loyalty: {len(CLASSIFIER_FEATURES)}) ===")
print(f"=== Regressors: {FINAL_N_ESTIMATORS_REG} est. (Full Features + LOYALTY: {len(REGRESSOR_FEATURES)}) ===")
print(f"=== Feature Segregation: ✓ ===")
print(f"=== Loyalty Features: REGRESSORS ONLY (for MAE stabilization) ===")

import time
total_start_time = time.time()

# Initialize lists to store predictions from each fold
all_clf_1w_preds = []
all_clf_2w_preds = []
all_reg_1w_preds = []
all_reg_2w_preds = []

print("\n=== Ensuring feature consistency between train and test ===")
# Ensure test sets have all required features
for col in CLASSIFIER_FEATURES:
    if col not in df_test_fe_clf.columns:
        print(f"  Adding missing classifier feature to test: {col}")
        if col in df_full_train_clf.columns:
            if df_full_train_clf[col].dtype.name == 'category':
                df_test_fe_clf[col] = pd.Categorical(
                    [df_full_train_clf[col].iloc[0]] * len(df_test_fe_clf),
                    categories=df_full_train_clf[col].cat.categories
                )
            else:
                df_test_fe_clf[col] = 0
        else:
            df_test_fe_clf[col] = 0

for col in REGRESSOR_FEATURES:
    if col not in df_test_fe_reg.columns:
        print(f"  Adding missing regressor feature to test: {col}")
        if col in df_full_train_reg.columns:
            if df_full_train_reg[col].dtype.name == 'category':
                df_test_fe_reg[col] = pd.Categorical(
                    [df_full_train_reg[col].iloc[0]] * len(df_test_fe_reg),
                    categories=df_full_train_reg[col].cat.categories
                )
            else:
                df_test_fe_reg[col] = 0
        else:
            df_test_fe_reg[col] = 0

for fold_idx, current_seed in enumerate(ENSEMBLE_SEEDS):
    print(f"\n{'='*60}")
    print(f"--- Training Fold {fold_idx + 1}/{NUM_ENSEMBLE_FOLDS} (Seed: {current_seed}) ---")
    fold_start_time = time.time()

    set_all_seeds(current_seed)
    np.random.seed(current_seed)
    random.seed(current_seed)

    try:
        # Train classifiers with CLEAN features (NO loyalty)
        clf_1w = train_reproducible_lgbm(df_full_train_clf, 'Target_purchase_next_1w',
                                        True, CLF_1W_PARAMS, current_seed,
                                        CLASSIFIER_FEATURES, REGRESSOR_FEATURES)
        clf_2w = train_reproducible_lgbm(df_full_train_clf, 'Target_purchase_next_2w',
                                        True, CLF_2W_PARAMS, current_seed,
                                        CLASSIFIER_FEATURES, REGRESSOR_FEATURES)

        # Train regressors with FULL features + LOYALTY
        reg_1w = train_reproducible_lgbm(df_full_train_reg, 'Target_qty_next_1w',
                                        False, REG_1W_PARAMS, current_seed,
                                        CLASSIFIER_FEATURES, REGRESSOR_FEATURES)
        reg_2w = train_reproducible_lgbm(df_full_train_reg, 'Target_qty_next_2w',
                                        False, REG_2W_PARAMS, current_seed,
                                        CLASSIFIER_FEATURES, REGRESSOR_FEATURES)

        # Store predictions
        all_clf_1w_preds.append(clf_1w.predict_proba(df_test_fe_clf[CLASSIFIER_FEATURES])[:, 1])
        all_clf_2w_preds.append(clf_2w.predict_proba(df_test_fe_clf[CLASSIFIER_FEATURES])[:, 1])
        all_reg_1w_preds.append(reg_1w.predict(df_test_fe_reg[REGRESSOR_FEATURES]).clip(min=0))
        all_reg_2w_preds.append(reg_2w.predict(df_test_fe_reg[REGRESSOR_FEATURES]).clip(min=0))

        fold_elapsed = time.time() - fold_start_time
        print(f"  ✓ Fold {fold_idx + 1} completed in {fold_elapsed:.1f} seconds!")

    except Exception as e:
        print(f"  Error in fold {fold_idx + 1}: {e}")
        import traceback
        traceback.print_exc()
        continue

# Average predictions across all folds
print(f"\nAveraging predictions across {len(all_clf_1w_preds)} folds...")

if all_clf_1w_preds:
    df_test_fe_clf['Target_purchase_next_1w'] = np.mean(all_clf_1w_preds, axis=0)
    df_test_fe_clf['Target_purchase_next_2w'] = np.mean(all_clf_2w_preds, axis=0)
    df_test_fe_reg['pred_qty_1w_raw'] = np.mean(all_reg_1w_preds, axis=0)
    df_test_fe_reg['pred_qty_2w_raw'] = np.mean(all_reg_2w_preds, axis=0)

    # Copy purchase predictions to regressor dataframe for power scaling
    df_test_fe_reg['Target_purchase_next_1w'] = df_test_fe_clf['Target_purchase_next_1w']
    df_test_fe_reg['Target_purchase_next_2w'] = df_test_fe_clf['Target_purchase_next_2w']
else:
    print("ERROR: No predictions were generated!")
    sys.exit(1)

total_elapsed = time.time() - total_start_time
print(f"\n{'='*60}")
print(f"✅ ENSEMBLE COMPLETE! Total training time: {total_elapsed:.1f} seconds")
print(f"Average time per fold: {total_elapsed/NUM_ENSEMBLE_FOLDS:.1f} seconds")
print(f"Feature Segregation Strategy:")
print(f"  - Classifiers: {len(CLASSIFIER_FEATURES)} clean features (AUC focused, NO loyalty)")
print(f"  - Regressors: {len(REGRESSOR_FEATURES)} full features (MAE focused + LOYALTY)")
print(f"  - Loyalty Features: {len([f for f in REGRESSOR_FEATURES if 'cp_' in f or 'qty_' in f])} features for regressors only")
print(f"{'='*60}")

### 24. Apply Improved Adaptive Power Scaling

This section applies a post-processing step called Adaptive Power Scaling to the predicted quantities. It defines a function that adjusts a power exponent based on the purchase probability: lower probabilities lead to a higher exponent, effectively reducing the predicted quantity further. This is designed to refine the quantity predictions and potentially improve MAE, especially for low-probability purchases.

In [ ]:
# ============================================================
# 8. IMPROVED Adaptive Power Scaling
# ============================================================

print("\n--- Applying Adaptive Power Scaling (using regressor dataframe) ---")

def adaptive_power_scaling(purchase_prob, recency_weeks=26):
    """Adaptive power scaling based on purchase probability."""
    if purchase_prob < 0.01:
        return 1.8
    elif purchase_prob < 0.1:
        return 1.5
    elif purchase_prob < 0.3:
        return 1.25
    elif purchase_prob < 0.5:
        return 1.1
    else:
        return 1.05

# Apply adaptive scaling
df_test_fe_reg['power_scale_1w'] = df_test_fe_reg['Target_purchase_next_1w'].apply(adaptive_power_scaling)
df_test_fe_reg['power_scale_2w'] = df_test_fe_reg['Target_purchase_next_2w'].apply(adaptive_power_scaling)

# Apply power scaling
df_test_fe_reg['Target_qty_next_1w'] = df_test_fe_reg.apply(
    lambda row: row['Target_purchase_next_1w'] ** row['power_scale_1w'] * row['pred_qty_1w_raw'],
    axis=1
)

df_test_fe_reg['Target_qty_next_2w'] = df_test_fe_reg.apply(
    lambda row: row['Target_purchase_next_2w'] ** row['power_scale_2w'] * row['pred_qty_2w_raw'],
    axis=1
)

### 25. Perform Final Post-processing

This cell applies final adjustments to the predicted quantities. It clips the `Target_qty_next_1w` and `Target_qty_next_2w` values to ensure they are non-negative and do not exceed a reasonable upper bound (1.5 times the maximum quantity in the training data). It also cleans up temporary columns (`pred_qty_1w_raw`, `power_scale_1w`, etc.) used during the prediction and scaling process, tidying the DataFrame.

In [ ]:
# ============================================================
# 9. FINAL POST-PROCESSING
# ============================================================

print("\n--- Final Post-processing ---")

# Apply clipping
df_test_fe_reg['Target_qty_next_1w'] = df_test_fe_reg['Target_qty_next_1w'].clip(
    lower=0,
    upper=MAX_QTY_TRAIN * 1.5
)

df_test_fe_reg['Target_qty_next_2w'] = df_test_fe_reg['Target_qty_next_2w'].clip(
    lower=0,
    upper=MAX_QTY_TRAIN * 1.5
)

# Clean up temporary columns
df_test_fe_reg = df_test_fe_reg.drop(
    columns=['pred_qty_1w_raw', 'pred_qty_2w_raw', 'power_scale_1w', 'power_scale_2w'],
    errors='ignore'
)

### 26. Generate Submission File

This cell constructs the final submission DataFrame. It combines the `ID` from the test set with the predicted purchase probabilities and quantities for both week 1 and week 2. It includes a check to ensure no duplicate IDs exist, which is critical for valid submissions. The resulting DataFrame is then ready to be saved as the submission file.

In [ ]:
# ============================================================
# 10. Generate Submission File
# ============================================================

submission = pd.DataFrame({
    'ID': df_test_fe_clf['ID'],
    'Target_purchase_next_1w': df_test_fe_clf['Target_purchase_next_1w'],
    'Target_qty_next_1w': df_test_fe_reg['Target_qty_next_1w'],
    'Target_purchase_next_2w': df_test_fe_clf['Target_purchase_next_2w'],
    'Target_qty_next_2w': df_test_fe_reg['Target_qty_next_2w']
})

if len(submission) == 275796 and submission['ID'].nunique() == 275796:
    print("✅ Perfect! No duplicates.")
else:
    print(f"⚠️ Cleaning duplicates...")
    submission = submission.drop_duplicates(subset=['ID'], keep='first')
    print(f"After cleaning: {len(submission)} rows")

### 27. Print Submission Preview and Save File

This final output cell displays a preview of the generated `submission` DataFrame, showing the first few rows of the predictions. It then saves the complete submission DataFrame to a CSV file named 'Monday_very_best.csv', ensuring it's in the correct format for submission. A summary of the model's key configurations and overall training time is also printed.

In [ ]:
print("\n✅ FINAL SUBMISSION PREVIEW:")
print(submission.head())

submission_file = 'Monday_very_best.csv'
submission.to_csv(submission_file, index=False)
print(f"\n✅ Submission file saved as '{submission_file}'")

# Final summary
print("\n" + "="*60)
print("SUMMARY:")
print(f"  - Random Seed: {RANDOM_SEED}")
print(f"  - Ensemble Folds: {NUM_ENSEMBLE_FOLDS}")
print(f"  - Classifier Estimators: {FINAL_N_ESTIMATORS_CLF}")
print(f"  - Regressor Estimators: {FINAL_N_ESTIMATORS_REG}")
print(f"  - Classifier Features: {len(CLASSIFIER_FEATURES)} (clean, AUC-focused, NO loyalty)")
print(f"  - Regressor Features: {len(REGRESSOR_FEATURES)} (full, MAE-focused + LOYALTY)")
print(f"  - Loyalty Features: REGRESSORS ONLY ({len([f for f in REGRESSOR_FEATURES if 'cp_' in f])} features)")
print(f"  - Parallel Training: Enabled (n_jobs=-1)")
print(f"  - Total Training Time: {total_elapsed:.1f}s")
print("="*60)

### 28. Display Submission Descriptive Statistics

This cell provides a quick statistical overview of the predicted values in the `submission` DataFrame. It calculates and displays key descriptive statistics (count, mean, std, min, 25%, 50%, 75%, max) for each of the target prediction columns (`Target_purchase_next_1w`, `Target_qty_next_1w`, etc.). This helps to understand the distribution and range of the model's final predictions.

In [ ]:
submission.describe()

,Target_purchase_next_1w,Target_qty_next_1w,Target_purchase_next_2w,Target_qty_next_2w
count,2.757960e+05,2.757960e+05,2.757960e+05,2.757960e+05
mean,3.554043e-02,1.943754e-01,4.077185e-02,4.382540e-01
std,1.449774e-01,3.320831e+00,1.618116e-01,7.583263e+00
min,3.985206e-07,0.000000e+00,1.512117e-08,0.000000e+00
25%,5.860068e-05,5.186002e-09,1.373687e-05,6.495016e-10
50%,3.075893e-04,1.109441e-07,9.203750e-05,1.981095e-08
75%,2.324699e-03,4.166855e-06,1.095583e-03,1.666623e-06
max,9.993954e-01,4.608296e+02,9.999798e-01,1.057724e+03
